Sean Bergan

Referring to `Python_exploring_h5ad_files.ipynb` and the [Scanpy Preprocessing and clustering tutorial](https://scanpy.readthedocs.io/en/stable/tutorials/basics/clustering.html#nearest-neighbor-graph-construction-and-visualization).

**This notebook includes:**
- Reading in h5ad files in our dataset
- Figuring out variable names in our adata and seeing if using ad.concat() + adding a `sample` column, vs. using sc.concat(), affects things downstream
- Exploring adata
- Examining the cell labels in the cell metadata (adata.obs['AIFI_L2']) to see if any labels are underrepresented in any of the drug treatments
     - Visualizing with bar charts and heatmaps
-  Scanpy preprocessing, dimensionality reduction (PCA, UMAPs) and clustering

In [1]:
# Core libraries
import hisepy
import numpy as np
import scanpy as sc
import anndata as ad
import pandas as pd

import seaborn as sns
import matplotlib.pyplot as plt

/home/workspace/environment/pythonscrna12/lib/python3.13/site-packages/leidenalg/VertexPartition.py:413: SyntaxWarning: invalid escape sequence '\m'
  .. math:: Q = \\frac{1}{m} \\sum_{ij} \\left(A_{ij} - \\frac{k_i^\mathrm{out} k_j^\mathrm{in}}{m} \\right)\\delta(\\sigma_i, \\sigma_j),
/home/workspace/environment/pythonscrna12/lib/python3.13/site-packages/leidenalg/VertexPartition.py:788: SyntaxWarning: invalid escape sequence '\m'
  .. math:: Q = \\sum_{ij} \\left(A_{ij} - \\gamma \\frac{k_i^\mathrm{out} k_j^\mathrm{in}}{m} \\right)\\delta(\\sigma_i, \\sigma_j),
/home/workspace/environment/pythonscrna12/lib/python3.13/site-packages/leidenalg/Optimiser.py:27: SyntaxWarning: invalid escape sequence '\g'
  implementation therefore does not guarantee subpartition :math:`\gamma`-density.
/home/workspace/environment/pythonscrna12/lib/python3.13/site-packages/leidenalg/Optimiser.py:346: SyntaxWarning: invalid escape sequence '\s'
  .. math:: Q = \sum_k \\lambda_k Q_k.


## Read in our processed .h5ad file

In [2]:
# Updated methodology for reading in adata:

# # for old h5id file name:
# uuid_list = ['a121aba6-89ae-4532-aa37-7e4fcf9b05e4']

uuid_list = ['fa51fd69-eb7c-4bd3-b80a-b8dab2603043']


# have to read it in as a list
files_list = hisepy.cache_files(uuid_list)
print(files_list)
# file list is only one file, so we can just access that for our adata
adata = sc.read_h5ad(files_list[0])

['/home/workspace/input/1742749117/2025_bgmp/fa51fd69-eb7c-4bd3-b80a-b8dab2603043/berkelium-silicon-hydrogen/il6_jak-stat_paired_formulations.h5ad']


## DEG for formulations
Let's do DEG with each formulation pair across all cell types

In [3]:
form_pairs = []

# unique_treatments = unique_treatments[unique_treatments['clean_drug_name'] == 'DMSO Control']

for n, formulation in enumerate(adata.obs['clean_drug_name'].cat.categories):
    if 'control' not in formulation:
        # if even (0 indexing), add a new entry to the list of lists
        print(f'n = {n}, formulation = {formulation}')
        if n % 2 == 0:
            form_pairs.append([formulation])
        # if odd, append to the list we just made
        else:
            # hard coding the -2 after we transitioned over to 'clean_drug_name' being a categorical
            # to account for the first 2 being our controls
            form_pairs[(n-2) // 2].append(formulation)

print(form_pairs)

n = 2, formulation = Afatinib
n = 3, formulation = Afatinib dimaleate
n = 4, formulation = Baricitinib
n = 5, formulation = Baricitinib phosphate
n = 6, formulation = Canertinib
n = 7, formulation = Canertinib dihydrochloride
n = 8, formulation = Erlotinib
n = 9, formulation = Erlotinib hydrochloride
n = 10, formulation = Gefitinib
n = 11, formulation = Gefitinib hydrochloride
n = 12, formulation = NVP-BSK805
n = 13, formulation = NVP-BSK805 2HCl
n = 14, formulation = Ruxolitinib
n = 15, formulation = Ruxolitinib phosphate
n = 16, formulation = Tofacitinib citrate
n = 17, formulation = Tofacitinib
[['Afatinib', 'Afatinib dimaleate'], ['Baricitinib', 'Baricitinib phosphate'], ['Canertinib', 'Canertinib dihydrochloride'], ['Erlotinib', 'Erlotinib hydrochloride'], ['Gefitinib', 'Gefitinib hydrochloride'], ['NVP-BSK805', 'NVP-BSK805 2HCl'], ['Ruxolitinib', 'Ruxolitinib phosphate'], ['Tofacitinib citrate', 'Tofacitinib']]


In [4]:
# making a dictionary of subsetted data where the key is the cell type and the value is the anndata object
subsets_cells_form_pairs = {}

detection_cutoff = 0.01

for cell_type in adata.obs['AIFI_L2'].unique():
    # print(f'current cell type is {cell_type}')
    cell_type_subset = adata[adata.obs.AIFI_L2 == cell_type].copy()
    for form_pair in form_pairs:
        form_1, form_2 = form_pair

        # filtering out genes that are expressed in too few of cells since they seem to be leading to weird results in DEG
        min_cells = int(cell_type_subset.shape[0] * detection_cutoff)
    
        sc.pp.filter_genes(
            cell_type_subset,
            min_cells = min_cells
        )
        
        treatment_subset = cell_type_subset[cell_type_subset.obs["clean_drug_name"].isin(form_pair)].copy()
        
        sc.tl.rank_genes_groups(
            adata = treatment_subset,
            groupby = "clean_drug_name",
            reference = form_1,
            method = "wilcoxon",
            layer = "log_transformed"
        )
        subsets_cells_form_pairs[(cell_type, tuple(form_pair))] = treatment_subset

# print(subsets_cells)

# figure out grabbing the data frames from these -- adata.to_df()

# get_rank

In [5]:
concat_form_pairs_df_list = []

for (cell_type, form_pair), subset_adata in subsets_cells_form_pairs.items():
    subset_df = sc.get.rank_genes_groups_df(subset_adata, group=None)
    # add columns for cell type and formula pair
    subset_df['cell_type'] = cell_type
    form_1, form_2 = form_pair
    subset_df['form_1'] = form_1
    subset_df['form_2'] = form_2

    # add subset to the list to be concatenated
    concat_form_pairs_df_list.append(subset_df)

form_pairs_df = pd.concat(concat_form_pairs_df_list)

In [6]:
print(form_pairs_df)
form_pairs_df.to_csv('head-to-head_wilcoxon_deg_results.csv')

        names    scores  logfoldchanges         pvals     pvals_adj  \
0        CD44  9.302001        0.051674  1.378264e-20  1.842739e-17   
1         CD2  8.935438        0.094797  4.055687e-19  2.711227e-16   
2     S100A10  7.404135        0.028103  1.320082e-13  5.673898e-11   
3         LTB  7.370687        0.086275  1.697501e-13  5.673898e-11   
4        FYB1  6.936693        0.088145  4.013839e-12  1.073301e-09   
...       ...       ...             ...           ...           ...   
1324     SELL -2.734802       -0.350373  6.241792e-03  6.380187e-01   
1325     AKT3 -2.773036       -0.285096  5.553597e-03  6.150609e-01   
1326    PRKCQ -2.781397       -0.195473  5.412555e-03  6.150609e-01   
1327   RIPOR2 -2.910409       -0.245898  3.609560e-03  6.150609e-01   
1328    MALT1 -3.554741       -0.397806  3.783511e-04  2.514143e-01   

                cell_type               form_1              form_2  
0      CD4 Central Memory             Afatinib  Afatinib dimaleate  
1      CD

## Uploading files

In [7]:
# starting off with a function to make unique destination strings with periodic table elements
def element_id(n = 3):
    import periodictable
    from random import randrange
    rand_el = []
    
    for i in range(n):
        el = randrange(0,118)
        rand_el.append(periodictable.elements[el].name)
        
    rand_str = '-'.join(rand_el)
    return rand_str

In [8]:
# need unique title for upload, using UTC time
from datetime import datetime, timezone

utc_current = datetime.now(timezone.utc)
print(utc_current)

# todo: should probably clean up the names here of these csvs... just replace "." with "_" in the threshold?
files_to_upload = [
    "head-to-head_wilcoxon_deg_results.csv"
]

hisepy.upload_files(
    files = files_to_upload,
    study_space_id = "c8a94b84-b0b7-40a9-980b-81a63ad6e115",
    title = f"form_pairs_deg_{utc_current}",
    input_file_ids = uuid_list,
    destination = element_id()
)

2026-02-17 20:41:48.318750+00:00
checking if conda environment can compile...
creating temp conda environment...
temp conda environment created successfully, now packing...


{'Message': 'General Okay-ness',
 'VisualizationId': '00000000-0000-0000-0000-000000000000',
 'AbstractionId': '00000000-0000-0000-0000-000000000000',
 'TraceId': '41fefa65-6a91-4f94-a63e-51b6ee96892a',
 'ProcessId': '0e76a23a-8c34-451b-94a3-ba76ef14478c',
 'WorkflowId': 'c8b95d2c-e8a6-49db-9fa4-20f50cba0bba',
 'FileIds': ['7ae9b75f-6013-4540-beff-c086448f9115']}